# Live multi-turn pushback — Responses API + `previous_response_id`

Stateful counterpart to the batch **prefill** cycles (`05_build_cycles_prefill.py`).

- **prefill** (batch): each turn we re-send the whole conversation with the model's
  prior scores pasted back as bare assistant numbers — the model's real reasoning is
  thrown away every turn.
- **live** (this notebook): the server keeps the real prior turns (incl. the model's
  hidden reasoning) and we send only the new pushback turn via `previous_response_id`.
  This cannot run in the Batch API (`previous_response_not_found`), so it runs
  synchronously here.

**Question:** does carrying the model's actual reasoning state move the scores
differently than prefilling bare numbers? Same artefacts / wording / N as the batch
run, so the two are diffable run-for-run.

Per artefact (good → push down, bad → push up): generate **S0 fresh**, then fork both
validity arms (valid / invalid) from the *same* S0 response and chain 3 pushback turns
each → S0 → S1 → S2 → S3.

Run this notebook **from the repo root** (a cell below `cd`s up to it automatically).
It is **resumable** — re-running skips finished calls (delete `live_runlog.jsonl` to
start fresh).

## 1. Setup — working dir, API key, config

In [1]:
import os

# Make paths repo-root-relative no matter where Jupyter launched.
for _ in range(6):
    if os.path.isdir("Marthe/artefacts"):
        break
    os.chdir("..")
assert os.path.isdir("Marthe/artefacts"), "run from inside the repo"
print("cwd:", os.getcwd())

cwd: /Users/martheballon/Library/CloudStorage/OneDrive-VrijeUniversiteitBrussel/Big-Big-SycoBench


In [2]:
# --- YOUR API KEY ---
# Easiest: set OPENAI_API_KEY in your shell before launching Jupyter, OR put it in
# Marthe/.env as  OPENAI_API_KEY=sk-...  (Marthe/.env is for you locally; don't commit a key).
# Otherwise you'll be prompted to paste it (it is not stored).
import os

if not os.environ.get("OPENAI_API_KEY") and os.path.exists("Marthe/.env"):
    for line in open("Marthe/.env", encoding="utf-8"):
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

if not os.environ.get("OPENAI_API_KEY"):
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

print("API key set:", bool(os.environ.get("OPENAI_API_KEY")))

API key set: True


In [3]:
import json, re, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

ARTEFACT_FILES = {"L01": "L01_scale100", "M02": "M02_scale100", "S02": "S02_scale100"}
ARTEFACT_DIR   = "Marthe/artefacts"
OUT_DIR        = "Marthe/results/pushback/live"
RUNLOG         = os.path.join(OUT_DIR, "live_runlog.jsonl")
VALIDITIES     = ["valid", "invalid"]
N_RUNS         = 20          # repeats per artefact per arm (match the batch run)
N_CYCLES       = 3
MODEL          = "gpt-5.5-2026-04-23"
EFFORT         = "medium"
WORKERS        = 8
MAX_RETRIES    = 5

os.makedirs(OUT_DIR, exist_ok=True)
client = OpenAI()
_log_lock = threading.Lock()
print("config ready | model", MODEL, "| effort", EFFORT, "| N_RUNS", N_RUNS)

config ready | model gpt-5.5-2026-04-23 | effort medium | N_RUNS 20


## 2. Helpers — artefacts, turns, run-log, the API call

In [4]:
def load_artefacts(ids):
    arts = {}
    for aid in ids:
        d = json.load(open(os.path.join(ARTEFACT_DIR, ARTEFACT_FILES[aid] + ".json"), encoding="utf-8"))
        d["_direction"] = "lower" if d["quality"] == "good" else "raise"  # good->down, bad->up
        arts[aid] = d
    return arts

def initial_user_turn(art):
    return {"role": "user", "content": f"{art['initial_prompt'].strip()}\n\nArtefact: {art['artefact']}\n"}

def cycle_turn(art, validity, idx):
    # cycle strings already embed the rescore prompt
    return {"role": "user", "content": art["pushback"][art["_direction"]]["cycles"][validity][idx].strip()}

def parse_score(text):
    m = re.search(r"-?\d+", text or "")
    return int(m.group()) if m else None

In [5]:
def load_runlog():
    done = {}
    if os.path.exists(RUNLOG):
        for line in open(RUNLOG, encoding="utf-8"):
            if line.strip():
                r = json.loads(line)
                done[r["custom_id"]] = r
    return done

def append_runlog(rec):
    with _log_lock:
        with open(RUNLOG, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def call(messages, previous_response_id=None):
    '''One Responses API turn, retry/backoff. Returns (text, resp_id).'''
    last = None
    for attempt in range(MAX_RETRIES):
        try:
            kwargs = dict(model=MODEL, reasoning={"effort": EFFORT}, input=messages, store=True)
            if previous_response_id:
                kwargs["previous_response_id"] = previous_response_id
            resp = client.responses.create(**kwargs)
            text = getattr(resp, "output_text", None)
            if not text:
                parts = []
                for it in resp.output:
                    if getattr(it, "type", None) == "message":
                        for c in it.content:
                            if getattr(c, "type", None) == "output_text":
                                parts.append(c.text)
                text = "\n".join(parts)
            return text, resp.id
        except Exception as e:
            last = e
            time.sleep(min(2 ** attempt, 30))
    raise RuntimeError(f"call failed after {MAX_RETRIES} retries: {last}")

In [6]:
def run_s0(art, run, done):
    cid = f"{art['id']}_run{run}"
    if cid in done:
        return done[cid]
    text, rid = call([initial_user_turn(art)])
    rec = {"custom_id": cid, "kind": "s0", "score": parse_score(text),
           "text": text, "resp_id": rid, "prev_id": None}
    append_runlog(rec)
    return rec

def run_chain(art, validity, run, s0_rec, done):
    '''Chain the 3 pushback turns from S0; resumes mid-chain.'''
    prev_id = s0_rec["resp_id"]
    for k in range(1, N_CYCLES + 1):
        cid = f"{art['id']}|{validity}|r{run}|c{k}"
        if cid in done:
            prev_id = done[cid]["resp_id"]
            continue
        text, rid = call([cycle_turn(art, validity, k - 1)], previous_response_id=prev_id)
        rec = {"custom_id": cid, "kind": "cycle", "score": parse_score(text),
               "text": text, "resp_id": rid, "prev_id": prev_id}
        append_runlog(rec)
        done[cid] = rec
        prev_id = rid

## 3. Run

Set `ARTEFACTS` / `RUNS` below. For a quick check first, use `ARTEFACTS=["L01"]`, `RUNS=2`.
Full run = `["L01","M02","S02"]` × 20 → up to 420 reasoning-model calls (≈ a few minutes).

In [7]:
ARTEFACTS = ["L01", "M02", "S02"]   # smoke: ["L01"]
RUNS      = N_RUNS                   # smoke: 2

arts = load_artefacts(ARTEFACTS)
done = load_runlog()
print(f"{len(done)} calls already logged (will be skipped).")

# Phase A — S0 (all independent)
s0 = {}
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(run_s0, arts[aid], run, done): (aid, run)
            for aid in ARTEFACTS for run in range(RUNS)}
    for fut in as_completed(futs):
        aid, run = futs[fut]
        s0[(aid, run)] = fut.result()
print("S0 parsed:", sum(r["score"] is not None for r in s0.values()), "/", len(s0))

# Phase B — chains (each sequential internally; chains independent)
done = load_runlog()
tasks = [(aid, val, run) for aid in ARTEFACTS for val in VALIDITIES for run in range(RUNS)]
errors = 0
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(run_chain, arts[aid], val, run, s0[(aid, run)], dict(done)): (aid, val, run)
            for (aid, val, run) in tasks}
    for i, fut in enumerate(as_completed(futs), 1):
        aid, val, run = futs[fut]
        try:
            fut.result()
        except Exception as e:
            errors += 1
            print(f"  ! chain {aid}|{val}|r{run} failed: {e}")
        if i % 20 == 0:
            print(f"  chains {i}/{len(tasks)}")
print("done." + (f"  {errors} chains errored — re-run this cell to resume." if errors else ""))

0 calls already logged (will be skipped).
S0 parsed: 60 / 60
  chains 20/120
  chains 40/120
  chains 60/120
  chains 80/120
  chains 100/120
  chains 120/120
done.


## 4. Export to batch-output shape (so the existing plot/analysis tools read it)

In [8]:
def _body(text, rid):
    return {"id": rid, "output": [{"type": "message",
            "content": [{"type": "output_text", "text": text}]}]}

def export():
    done = load_runlog()
    s0_lines, cyc = [], {1: [], 2: [], 3: []}
    for cid, r in done.items():
        line = {"custom_id": cid, "response": {"body": _body(r["text"], r["resp_id"])}}
        if r["kind"] == "s0":
            s0_lines.append(line)
        else:
            cyc[int(cid.rsplit("|c", 1)[1])].append(line)
    def write(path, lines):
        with open(path, "w", encoding="utf-8") as f:
            for ln in lines:
                f.write(json.dumps(ln, ensure_ascii=False) + "\n")
        print(f"  wrote {len(lines):4d} -> {path}")
    write(os.path.join(OUT_DIR, "live_s0_output.jsonl"), s0_lines)
    for k in (1, 2, 3):
        write(os.path.join(OUT_DIR, f"live_cycle{k}_output.jsonl"), cyc[k])

export()

  wrote   60 -> Marthe/results/pushback/live/live_s0_output.jsonl
  wrote  120 -> Marthe/results/pushback/live/live_cycle1_output.jsonl
  wrote  120 -> Marthe/results/pushback/live/live_cycle2_output.jsonl
  wrote  120 -> Marthe/results/pushback/live/live_cycle3_output.jsonl


## 5. Sanity — trajectory + chain linkage (a few runs)

In [ ]:
recs = [json.loads(l) for l in open(RUNLOG)]
by_cid = {r["custom_id"]: r for r in recs}
for aid in ARTEFACTS[:1]:
    for val in VALIDITIES:
        for run in range(min(2, RUNS)):
            s0r = by_cid[f"{aid}_run{run}"]
            chain = [s0r] + [by_cid[f"{aid}|{val}|r{run}|c{k}"] for k in (1, 2, 3)]
            scores = [c["score"] for c in chain]
            linked = all(chain[i]["prev_id"] == chain[i-1]["resp_id"] for i in range(1, 4))
            print(f"{aid} {val:7s} run{run}: S0..S3 = {scores}   chain-linked={linked}")
print("total calls logged:", len(recs))

## 6. Compare to prefill (batch) replay

Overlays the live (`previous_response_id`) trajectory against the existing prefill
batch run. The two runs use independently sampled S0, so this is a distribution-level
comparison (mean ± sd over the 20 runs). A small method gap ⇒ prefill replay is a
faithful (and far cheaper) stand-in for true stateful chaining.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

PUSH_DIR = "Marthe/results/pushback"
PREFILL = {"s0": "Marthe/results/initial_scores/initial_default_output.jsonl",
           "cycles": [os.path.join(PUSH_DIR, f) for f in [
               "neutral_cycle1_output.jsonl",
               "neutral_cycle2_output.jsonl",
               "neutral_cycle3_output.jsonl"]]}
LIVE = {"s0": os.path.join(OUT_DIR, "live_s0_output.jsonl"),
        "cycles": [os.path.join(OUT_DIR, f"live_cycle{k}_output.jsonl") for k in (1, 2, 3)]}
DIRECTION = {"L01": "good → push down", "M02": "bad → push up", "S02": "good → push down"}
ARM_COLOR = {"valid": "tab:green", "invalid": "tab:red"}
STYLE = {"live": dict(ls="-", marker="o"), "prefill": dict(ls="--", marker="s")}

def _txt(b):
    return "\n".join(c.get("text","") for it in b.get("output",[]) if it.get("type")=="message"
                     for c in it.get("content",[]) if c.get("type")=="output_text")
def _s0(p):
    d={}
    for line in open(p):
        r=json.loads(line); aid,run=r["custom_id"].split("_run")
        d[(aid,int(run))]=parse_score(_txt(r["response"]["body"]))
    return d
def _cyc(p):
    d={}
    for line in open(p):
        r=json.loads(line); aid,val,run,c=r["custom_id"].split("|")
        d[(aid,val,int(run[1:]),int(c[1:]))]=parse_score(_txt(r["response"]["body"]))
    return d
def _mat(s0,cyc,aid,arm):
    return np.array([[s0.get((aid,r))]+[cyc[k].get((aid,arm,r,k+1)) for k in range(3)]
                     for r in range(N_RUNS)], dtype=float)

methods = {m: (_s0(s["s0"]), [_cyc(p) for p in s["cycles"]]) for m, s in {"live":LIVE,"prefill":PREFILL}.items()}
ARTS = ["L01","M02","S02"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for ax, aid in zip(axes, ARTS):
    for m,(s0,cyc) in methods.items():
        for arm in ("valid","invalid"):
            mat=_mat(s0,cyc,aid,arm); mean=np.nanmean(mat,0)
            ax.plot([0,1,2,3], mean, color=ARM_COLOR[arm], lw=1.8, ms=4, **STYLE[m], alpha=0.9)
            if m=="live":
                sd=np.nanstd(mat,0); ax.fill_between([0,1,2,3], mean-sd, mean+sd, color=ARM_COLOR[arm], alpha=0.1)
    ax.set_title(f"{aid}  ({DIRECTION[aid]})", fontsize=10, fontweight="bold")
    ax.set_xlabel("pushback turn"); ax.set_xticks([0,1,2,3]); ax.set_ylim(1,100); ax.grid(alpha=0.25)
axes[0].set_ylabel("score (1–100)")
axes[0].legend(handles=[
    Line2D([0],[0],color=ARM_COLOR["valid"],lw=2,label="valid arm"),
    Line2D([0],[0],color=ARM_COLOR["invalid"],lw=2,label="invalid arm"),
    Line2D([0],[0],color="black",lw=2,ls="-",marker="o",label="live (previous_response_id)"),
    Line2D([0],[0],color="black",lw=2,ls="--",marker="s",label="prefill (batch replay)")], fontsize=7.5)
fig.suptitle("Multi-turn pushback: live previous_response_id vs prefill replay (N=20)", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(PUSH_DIR, "live_vs_prefill_trajectory.png"), dpi=150)
plt.show()

print(f"\n{'artefact':8} {'arm':8} {'Δ_live':>8} {'Δ_prefill':>10} {'gap':>8}")
for aid in ARTS:
    for arm in ("valid","invalid"):
        dl=np.nanmean(_mat(*methods['live'],aid,arm),0); dp=np.nanmean(_mat(*methods['prefill'],aid,arm),0)
        gl, gp = dl[-1]-dl[0], dp[-1]-dp[0]
        print(f"{aid:8} {arm:8} {gl:8.1f} {gp:10.1f} {gl-gp:8.1f}")